In [10]:
1.60217662e-19*3.4e-4

5.447400508000001e-23

In [12]:
import numpy as np

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------
e   = 1.60217662e-19        # C
h   = 6.62607004e-34        # J*s
R_Q = h / (4 * e**2)        # Ohm
kOhm = 1e3                  # Ohm

# Aluminium gap: 2Δ ≈ 3.4e-4 eV → Δ
Delta_2_Al = 2*0.163e-3 * e     # J, 2Δ
Delta_Al   = Delta_2_Al / 2 # J, Δ
# Delta_Al = 9.276e-23 /2  # J, refined value

# Junction capacitance density (given)
C_J_nom = 45e-15            # F / µm^2

# Series resistor used during measurement
R_series = 100.0            # kOhm


# ------------------------------------------------------------
# Raw measured resistances from the notebook
# ------------------------------------------------------------

# Chain pads (flattened list)
R_chain_meas = np.array([
    127.4, 127.4, 127.5, 127.2, 127.4, 127.4,
    127.7, 128.0, 127.5, 127.4, 127.5, 127.4,
    127.4, 127.6, 127.4, 127.4, 127.6, 127.6
])

# Single-JJ pads
R_jj_meas = np.array([
    [132.1, 134.5, 133.5],
    [133.0, 100.5, 135.8],
    [132.1, 100.9, 132.0],
    [133.0, 133.4, 132.0],
    [133.9, 132.7, 132.4],
    [133.0, 133.1, 133.5],
]).ravel()


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def filter_good_R(values,
                  R_series=100.0,
                  dead_tol=5.0,
                  z_thresh=3.0):
    """
    Remove 'dead' devices (~R_series) and robustly filter out outliers.
    Returns an array of good pad resistances.
    """
    vals = np.asarray(values, dtype=float)

    # Drop 'dead' ones close to the series resistor
    alive = vals[np.abs(vals - R_series) > dead_tol]
    if alive.size == 0:
        raise RuntimeError("No alive devices after dead filtering")

    # Robust outlier removal via MAD
    median = np.median(alive)
    mad = np.median(np.abs(alive - median))
    if mad == 0:
        return alive

    z = 0.6745 * (alive - median) / mad
    return alive[np.abs(z) < z_thresh]


def EJ_div_h_from_Rn(R_n, Delta=Delta_Al):
    """
    Ambegaokar–Baratoff (T~0):
      EJ/h = (R_Q / R_n) * (Δ / (2h)).

    Input:
        R_n   [Ohm]
        Delta [J]

    Returns:
        EJ_div_h [Hz]
    """
    R_n = np.asarray(R_n, dtype=float)
    return (R_Q / R_n) * (Delta / (2 * h))


def plasma_freq_from_Rn_and_area(R_n, area_um2,
                                 C_J_nom=C_J_nom,
                                 Delta=Delta_Al):
    """
    Compute plasma frequency for a JJ of given normal resistance and area.

      CJ = C_J_nom * area
      EC/h = e^2 / (2 CJ h)
      f_p  = sqrt(8 EJ/h * EC/h)
      ω_p  = 2π f_p
    """
    EJ_div_h = EJ_div_h_from_Rn(R_n, Delta=Delta)
    C_J = C_J_nom * area_um2
    EC_div_h = e**2 / (2 * C_J * h)
    EC_div_GHz = EC_div_h / 1e9
    EJ_div_GHz = EJ_div_h / 1e9
    print(f"EJ/h = {EJ_div_GHz:.3f} GHz, EC/h = {EC_div_GHz:.3f} GHz")
    f_p = np.sqrt(8.0 * EJ_div_h * EC_div_h)
    omega_p = 2 * np.pi * f_p
    return f_p, omega_p


# ------------------------------------------------------------
# Design parameters
# ------------------------------------------------------------

# Chain junction counts (you can change these)
N_long  = 5+3+5   # number of long junctions
N_short = 2+2+2    # number of short junctions
N_chain = N_long + N_short

# Chain junction sizes (µm)
a_long,  b_long  = 2.6, 0.28
a_short, b_short = 2.6, 0.27
area_long  = a_long  * b_long
area_short = a_short * b_short

# Single JJ size (µm)
a_JJ, b_JJ = 0.15, 0.15
area_JJ = a_JJ * b_JJ


# ------------------------------------------------------------
# 1) Process chain resistances (total, then per type)
# ------------------------------------------------------------
R_chain_good = filter_good_R(R_chain_meas, R_series=R_series)
R_chain_pad_avg = np.mean(R_chain_good)

# Subtract series resistor
R_chain_total = R_chain_pad_avg - R_series

# Assume same specific resistance ρ = R_n * A for all chain junctions:
#   R_chain_total = sum_j ρ / A_j
#                  = ρ (N_long / A_long + N_short / A_short)
rho = R_chain_total / (N_long / area_long + N_short / area_short)

R_n_long  = rho / area_long
R_n_short = rho / area_short


# ------------------------------------------------------------
# 2) Process single-JJ resistances
# ------------------------------------------------------------
R_jj_good = filter_good_R(R_jj_meas, R_series=R_series)
R_jj_pad_avg = np.mean(R_jj_good)
R_n_JJ = R_jj_pad_avg - R_series


# ------------------------------------------------------------
# 3) Plasma frequencies
# ------------------------------------------------------------
f_p_long,  omega_p_long  = plasma_freq_from_Rn_and_area(R_n_long * kOhm,  area_long)
f_p_short, omega_p_short = plasma_freq_from_Rn_and_area(R_n_short * kOhm, area_short)
f_p_JJ,    omega_p_JJ    = plasma_freq_from_Rn_and_area(R_n_JJ * kOhm,    area_JJ)


# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------
print("=== Chain resistances ===")
print(f"Average chain pad R          : {R_chain_pad_avg:.3f} Ω")
print(f"Total chain R (no 100 Ω)     : {R_chain_total:.3f} Ω")
print(f"R_n,long  (per long JJ)      : {R_n_long:.3f} Ω")
print(f"R_n,short (per short JJ)     : {R_n_short:.3f} Ω")
print()
print("=== Single JJ resistance ===")
print(f"Average single JJ pad R      : {R_jj_pad_avg:.3f} Ω")
print(f"R_n,single JJ (no 100 Ω)     : {R_n_JJ:.3f} Ω")
print()
print("=== Plasma frequencies ===")
print(f"Long chain JJ : f_p = {f_p_long/1e9:.3f} GHz, "
      f"ω_p = {omega_p_long/1e9:.3f} ×10^9 rad/s")
print(f"Short chain JJ: f_p = {f_p_short/1e9:.3f} GHz, "
      f"ω_p = {omega_p_short/1e9:.3f} ×10^9 rad/s")
print(f"Single JJ     : f_p = {f_p_JJ/1e9:.3f} GHz, "
      f"ω_p = {omega_p_JJ/1e9:.3f} ×10^9 rad/s")


EJ/h = 89.073 GHz, EC/h = 0.591 GHz
EJ/h = 85.892 GHz, EC/h = 0.613 GHz
EJ/h = 3.860 GHz, EC/h = 19.131 GHz
=== Chain resistances ===
Average chain pad R          : 127.444 Ω
Total chain R (no 100 Ω)     : 27.444 Ω
R_n,long  (per long JJ)      : 1.428 Ω
R_n,short (per short JJ)     : 1.481 Ω

=== Single JJ resistance ===
Average single JJ pad R      : 132.947 Ω
R_n,single JJ (no 100 Ω)     : 32.947 Ω

=== Plasma frequencies ===
Long chain JJ : f_p = 20.526 GHz, ω_p = 128.972 ×10^9 rad/s
Short chain JJ: f_p = 20.526 GHz, ω_p = 128.972 ×10^9 rad/s
Single JJ     : f_p = 24.305 GHz, ω_p = 152.715 ×10^9 rad/s
